In [ ]:
import pandas as pd
import ast
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.compose import ColumnTransformer 
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df = pd.read_csv('updated_aggregated_advertisers_dataset_v2.csv')


### Cleaning data of (unique_goals, order_goals) columns

In [ ]:
def clean_messy_list_string(x):
    if pd.isna(x) or x == "":
        return [] 
    
    x = str(x).upper()
    words = re.findall(r'[A-Z_]{3,}', x)
    clean_words = [word.lower() for word in words if word]
    return clean_words  

df['unique_goals'] = df['unique_goals'].apply(clean_messy_list_string)
df['order_goals'] = df['order_goals'].apply(clean_messy_list_string)


### Cleaning data of (postal_codes) for vectorization

In [ ]:
df['postal_codes'] = df['postal_codes'].apply(ast.literal_eval)

### checking order_goals and unique_goals rows similarity

In [ ]:
# Show which rows are the same
same_rows = df[df['order_goals'] == df['unique_goals']]
len(same_rows)

# Show which rows are different
different_rows = df[~(df['order_goals'] == df['unique_goals'])]
len(different_rows)

percentage_different = (len(different_rows) / len(df)) * 100
percentage_same = (len(same_rows) / len(df)) * 100

print(f"{percentage_different:.2f}% of the rows are different.")
print(f"{percentage_same:.2f}% of the rows are identical.")
print(f"{len(same_rows)} out of {len(df)} rows are identical.")

### BOW Vectorization of columns_to_use

In [ ]:
columns_to_use = ['unique_goals', 'unique_industries', 'states', 'cities', 
                  'favorite_city', 'favorite_state', 'advertiser_city', 'advertiser_state_code']

def preprocess_text_preserve_multiword(series):
    return series.astype(str) \
        .str.replace(r"[\[\]']", "", regex=True) \
        .str.replace(r"[&]", "", regex=True) \
        .str.split(",") \
        .apply(lambda items: ' '.join(
            re.sub(r'[^\w\s]', '', item).strip().replace(" ", "_")
            for item in items if item.strip()
        ))

def create_semantic_bow_vectors(df):
    geographic_cols = ['states', 'cities', 'favorite_city', 'favorite_state', 
                      'advertiser_city', 'advertiser_state_code']
    goal_cols = ['unique_goals']
    industry_cols = ['unique_industries']
    advertiser_name_cols = ['advertiser_name']
    
    vectors = {}
    vectorizers = {}
    
    # 1. Geographic BOW Vector
    geo_text = df[geographic_cols].apply(preprocess_text_preserve_multiword)
    geo_text_combined = geo_text.apply(lambda row: ' '.join(row), axis=1)
    geo_vectorizer = CountVectorizer(lowercase=True)
    geo_bow = geo_vectorizer.fit_transform(geo_text_combined)
    geo_features = [f"{name}" for name in geo_vectorizer.get_feature_names_out()]
    vectors['geographic'] = pd.DataFrame(geo_bow.toarray(), columns=geo_features, index=df.index)
    vectorizers['geographic'] = geo_vectorizer
    
    # 2. Goals BOW Vector
    goal_text = df[goal_cols].apply(preprocess_text_preserve_multiword)
    goal_text_combined = goal_text.apply(lambda row: ' '.join(row), axis=1)
    goal_vectorizer = CountVectorizer(lowercase=True)
    goal_bow = goal_vectorizer.fit_transform(goal_text_combined)
    goal_features = [f"{name}" for name in goal_vectorizer.get_feature_names_out()]
    vectors['goals'] = pd.DataFrame(goal_bow.toarray(), columns=goal_features, index=df.index)
    vectorizers['goals'] = goal_vectorizer
    
    # 3. Industry BOW Vector
    industry_text = df[industry_cols].apply(preprocess_text_preserve_multiword)
    industry_text_combined = industry_text.apply(lambda row: ' '.join(row), axis=1)
    industry_vectorizer = CountVectorizer(lowercase=True)
    industry_bow = industry_vectorizer.fit_transform(industry_text_combined)
    industry_features = [f"{name}" for name in industry_vectorizer.get_feature_names_out()]
    vectors['industry'] = pd.DataFrame(industry_bow.toarray(), columns=industry_features, index=df.index)
    vectorizers['industry'] = industry_vectorizer

    advertiser_text = df[advertiser_name_cols].apply(preprocess_text_preserve_multiword)
    advertiser_text_combined = advertiser_text.apply(lambda row: ' '.join(row), axis=1)
    advertiser_vectorizer = CountVectorizer(lowercase=True)
    advertiser_bow = advertiser_vectorizer.fit_transform(advertiser_text_combined)
    advertiser_features = [f"{name}" for name in advertiser_vectorizer.get_feature_names_out()]
    vectors['advertiser'] = pd.DataFrame(advertiser_bow.toarray(), columns=advertiser_features, index=df.index)
    vectorizers['advertiser'] = advertiser_vectorizer

    return vectors, vectorizers

# Create semantic vectors
semantic_vectors, semantic_vectorizers = create_semantic_bow_vectors(df)

print("Geographic BOW shape:", semantic_vectors['geographic'].shape)
print("Goals BOW shape:", semantic_vectors['goals'].shape) 
print("Industry BOW shape:", semantic_vectors['industry'].shape)
print("Advertiser BOW shape:", semantic_vectors['advertiser'].shape)

# Option 1: Use vectors separately for different analyses
geo_similarity = semantic_vectors['geographic']
goal_similarity = semantic_vectors['goals']
industry_similarity = semantic_vectors['industry']
advertiser_similarity = semantic_vectors['advertiser']

# Option 2: Combine all vectors (if needed for single model)
combined_semantic_bow = pd.concat([
    semantic_vectors['geographic'],
    semantic_vectors['goals'], 
    semantic_vectors['industry'],
    semantic_vectors['advertiser']
], axis=1)
geo_similarity

### Applying StandardScaler, MultiLabelBinarizerTransformer, OneHotEncoder

In [ ]:
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()

    def fit(self, X, y=None):
        return self.mlb.fit(X)

    def transform(self, X):
        return self.mlb.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.mlb.classes_

postal_code_encoder = MultiLabelBinarizerTransformer()
advertiser_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(transformers=[
    ('duration', StandardScaler(), ['avg_order_duration_days']),
    ('postal_codes', postal_code_encoder, 'postal_codes'),
    ('advertiser_zip', advertiser_encoder, ['advertiser_postal_code'])
], remainder='drop')

X_transformed = preprocessor.fit_transform(df)
duration_features = ['avg_order_duration_days']

postal_code_encoder.fit(df['postal_codes'])
postal_code_features = list(postal_code_encoder.mlb.classes_)

advertiser_features = list(preprocessor.named_transformers_['advertiser_zip'].get_feature_names_out(['advertiser_postal_code']))
all_feature_names = duration_features + postal_code_features + advertiser_features

df_transformed = pd.DataFrame(X_transformed, columns=all_feature_names)
df_transformed.shape

In [ ]:
df_transformed

### Concatinating both vector matrices 

In [ ]:
combined_semantic_bow = combined_semantic_bow.reset_index(drop=True)
df_transformed = df_transformed.reset_index(drop=True)

final_features_df = pd.concat([df_transformed, combined_semantic_bow], axis=1)
final_features_df.shape

In [ ]:
# After your training code that works, add this to save all transformers:
saved_transformers = {
    'semantic_vectorizers': semantic_vectorizers,
    'preprocessor': preprocessor,
    'all_feature_names': all_feature_names
}
print("All transformers saved successfully!")

In [ ]:
class LocationRecommendationSystem:
    
    def __init__(self, saved_transformers, feature_matrix, original_df):
        self.saved_transformers = saved_transformers
        self.feature_matrix = feature_matrix
        self.original_df = original_df
    
    def clean_messy_list_string(x):
        if pd.isna(x) or x == "":
            return [] 
        
        x = str(x).upper()
        words = re.findall(r'[A-Z_]{3,}', x)
        clean_words = [word.lower() for word in words if word]
        return clean_words  
    
    def preprocess_text_preserve_multiword(self, series):
        """Preprocess text while preserving multi-word phrases."""
        def process_item(item):
            if isinstance(item, list):
                # Join list items with spaces, then process
                text = ' '.join([str(x) for x in item])
            else:
                text = str(item)
            
            # Convert to lowercase and handle special characters
            text = text.lower()
            
            # Handle common patterns like "Apparel & Accessories" -> "apparel__accessories"
            # This matches the training data format shown in vocab
            text = text.replace(' & ', '__')  # "apparel & accessories" -> "apparel__accessories"
            text = text.replace('&', '__')    # Handle cases where & has no spaces
            text = text.replace(' ', '_')     # Replace remaining spaces with single underscore
            
            return text
        
        return series.apply(process_item)
    
    def normalize_industry(self, industry_value):
        """Normalize industry value to handle different formats."""
        if isinstance(industry_value, list):
            result = industry_value[0] if len(industry_value) > 0 else ""
            return result
        elif isinstance(industry_value, str):
            if industry_value.startswith('[') and industry_value.endswith(']'):
                try:
                    parsed = ast.literal_eval(industry_value)
                    if isinstance(parsed, list) and len(parsed) > 0:
                        result = parsed[0]
                        return result
                except:
                    match = re.search(r"'([^']+)'", industry_value)
                    if match:
                        result = match.group(1)
                        return result
            result = industry_value
            return result
        
        result = str(industry_value) if industry_value else ""
        return result
    
    def goals_match(self, row_goals, user_goals):
        """Check if goals match between two records."""
        if isinstance(row_goals, list) and isinstance(user_goals, list):
            row_set = set(row_goals)
            user_set = set(user_goals)
            match_result = row_set == user_set
            return match_result
        
        return False
    
    def build_user_input_record(self, advertiser_name, industry, duration, goals):
        # Process goals
        goals_cleaned = clean_messy_list_string(goals)
        
        # Normalize industry as string to match training data format
        if isinstance(industry, list):
            industry_normalized = industry[0] if len(industry) > 0 else "unknown"
        else:
            industry_normalized = str(industry) if industry else "unknown"
        
        return pd.DataFrame([{
            'avg_order_duration_days': duration,
            'postal_codes': [],
            'advertiser_postal_code': '00000',
            'unique_goals': goals_cleaned,
            'unique_industries': industry_normalized,
            'states': ['unknown_state'],
            'cities': ['unknown_city'],
            'favorite_city': ['unknown_city'],
            'favorite_state': ['unknown_state'],
            'advertiser_city': ['unknown_city'],
            'advertiser_state_code': ['unknown_state'],
            'advertiser_name': advertiser_name if advertiser_name else 'unknown_advertiser'
        }])
    
    def create_semantic_bow_vectors_inference(self, user_df):
        """Transform user input using pre-fitted vectorizers."""
        semantic_vectorizers = self.saved_transformers['semantic_vectorizers']
        
        geographic_cols = ['states', 'cities', 'favorite_city', 'favorite_state',
                          'advertiser_city', 'advertiser_state_code']
        goal_cols = ['unique_goals']
        industry_cols = ['unique_industries']
        advertiser_name_cols = ['advertiser_name']
        
        vectors = {}
        
        # Geographic BOW Vector
        geo_text = user_df[geographic_cols].apply(self.preprocess_text_preserve_multiword)
        geo_text_combined = geo_text.apply(lambda row: ' '.join(row), axis=1)
        geo_bow = semantic_vectorizers['geographic'].transform(geo_text_combined)
        geo_features = [f"{name}" for name in semantic_vectorizers['geographic'].get_feature_names_out()]
        vectors['geographic'] = pd.DataFrame(geo_bow.toarray(), columns=geo_features, index=user_df.index)
        
        # Goals BOW Vector
        goal_text = user_df[goal_cols].apply(self.preprocess_text_preserve_multiword)
        goal_text_combined = goal_text.apply(lambda row: ' '.join(row), axis=1)
        goal_bow = semantic_vectorizers['goals'].transform(goal_text_combined)
        goal_features = [f"{name}" for name in semantic_vectorizers['goals'].get_feature_names_out()]
        vectors['goals'] = pd.DataFrame(goal_bow.toarray(), columns=goal_features, index=user_df.index)
        
        # Industry BOW Vector
        industry_text = user_df[industry_cols].apply(self.preprocess_text_preserve_multiword)
        industry_text_combined = industry_text.apply(lambda row: ' '.join(row), axis=1)
        industry_bow = semantic_vectorizers['industry'].transform(industry_text_combined)
        industry_features = [f"{name}" for name in semantic_vectorizers['industry'].get_feature_names_out()]
        vectors['industry'] = pd.DataFrame(industry_bow.toarray(), columns=industry_features, index=user_df.index)
        
        # Advertiser BOW Vector
        advertiser_text = user_df[advertiser_name_cols].apply(self.preprocess_text_preserve_multiword)
        advertiser_text_combined = advertiser_text.apply(lambda row: ' '.join(row), axis=1)
        advertiser_bow = semantic_vectorizers['advertiser'].transform(advertiser_text_combined)
        advertiser_features = [f"{name}" for name in semantic_vectorizers['advertiser'].get_feature_names_out()]
        vectors['advertiser'] = pd.DataFrame(advertiser_bow.toarray(), columns=advertiser_features, index=user_df.index)
        
        return vectors
    
    def vectorize_user_input(self, user_df):
        # Create semantic vectors
        semantic_vectors = self.create_semantic_bow_vectors_inference(user_df)
        
        # Transform other features
        X_transformed = self.saved_transformers['preprocessor'].transform(user_df)
        df_transformed = pd.DataFrame(X_transformed, columns=self.saved_transformers['all_feature_names'])
        
        # Combine semantic vectors in the same order as training
        combined_semantic_bow = pd.concat([
            semantic_vectors['geographic'],
            semantic_vectors['goals'],
            semantic_vectors['industry'],
            semantic_vectors['advertiser']
        ], axis=1)
        
        # Reset indices and combine all features
        combined_semantic_bow = combined_semantic_bow.reset_index(drop=True)
        df_transformed = df_transformed.reset_index(drop=True)
        final_user_features = pd.concat([df_transformed, combined_semantic_bow], axis=1)
        
        return final_user_features.values
    
    def find_exact_matches(self, user_input_df, tolerance=1.0):
        user_goals = user_input_df['unique_goals'].iloc[0]
        user_industry = user_input_df['unique_industries'].iloc[0]
        user_advertiser = user_input_df['advertiser_name'].iloc[0]
        user_duration = user_input_df['avg_order_duration_days'].iloc[0]
        
        exact_matches = []
        
        for idx, row in self.original_df.iterrows():
            # Normalize industries for comparison
            row_industry = self.normalize_industry(row['unique_industries'])
            user_industry_norm = self.normalize_industry(user_industry)
            
            # Check all conditions
            goals_match_result = self.goals_match(row['unique_goals'], user_goals)
            industry_match = row_industry == user_industry_norm
            advertiser_match = row['advertiser_name'] == user_advertiser
            duration_match = abs(row['avg_order_duration_days'] - user_duration) < tolerance
            
            if goals_match_result and industry_match and advertiser_match and duration_match:
                exact_matches.append(idx)
        
        return exact_matches
    
    def get_similarity_recommendations(self, user_vector, top_k=5):
        
        print(user_vector.shape)
        print(self.feature_matrix.shape)
        similarities = cosine_similarity(user_vector, self.feature_matrix)[0]
        top_indices = similarities.argsort()[::-1][:top_k]
        
        recommendations = self.original_df.iloc[top_indices][['postal_codes', 'cities', 'states']].copy()
        similarity_scores = similarities[top_indices]
        
        return recommendations, similarity_scores
    
    def recommend_locations(self, advertiser_name, industry, duration, goals, top_k=5, prefer_exact_match=True):
        # Build user input record
        user_input_df = self.build_user_input_record(advertiser_name, industry, duration, goals)
        
        # Try exact matching first
        if prefer_exact_match:
            exact_matches = self.find_exact_matches(user_input_df)
            
            if len(exact_matches) > 0:
                recommendations = self.original_df.iloc[exact_matches][['postal_codes', 'cities', 'states']].head(top_k)
                return {
                    'recommendations': recommendations,
                    'method_used': 'exact_match',
                    'num_exact_matches': len(exact_matches),
                    'similarity_scores': None
                }
        
        # Fall back to similarity-based recommendations
        user_vector = self.vectorize_user_input(user_input_df)
        recommendations, similarity_scores = self.get_similarity_recommendations(user_vector, top_k)
        
        return {
            'recommendations': recommendations,
            'method_used': 'similarity_based',
            'num_exact_matches': 0,
            'similarity_scores': similarity_scores
        }


# Initialize the system with your saved transformers and data
recommendation_system = LocationRecommendationSystem(
    saved_transformers=saved_transformers,  # Your saved transformers dict
    feature_matrix=final_features_df.values,          # Your pre-computed feature matrix
    original_df=df                 # Your original training dataframe
)




In [ ]:
# Get recommendations
results = recommendation_system.recommend_locations(
    advertiser_name="hussnain",
    industry=['abxs'],
    duration=42,
    goals=["['ddd]"],
    top_k=5
)

results['recommendations']